## 웹서비스 흐름
1. 클라이언트가 `http://example.com` 같은 URL로 요청을 보낸다.
2. 웹서버가 HTML 문서를 응답한다.
3. 웹브라우저는 HTML을 파싱해서 화면에 렌더링한다.

### 정적 웹페이지 스크래핑 전 확인
1. 대상 사이트의 이용약관을 확인한다.
2. `robots.txt`를 확인하여 크롤러 접근 정책을 확인한다.
3. `robots.txt`는 크롤러에게 “이 경로는 수집하지 말아 달라”고 알려주는 표준 정책 파일이다.
4. 단, `robots.txt`가 허용한다고 해서 저작권, 개인정보, 이용약관 문제가 모두 해결되는 것은 아니다.
5. 짧은 시간에 반복 요청을 보내면 서버에 부담을 줄 수 있으므로 요청 간격과 수집 범위를 조절한다.

### 정적 웹페이지란?
- 서버가 응답한 HTML 안에 필요한 데이터가 이미 포함된 페이지이다.
- `requests`로 HTML을 가져온 뒤 `BeautifulSoup`으로 원하는 태그를 찾을 수 있다.
- JavaScript 실행 후에 데이터가 생기는 페이지는 정적 방식만으로 수집이 어려울 수 있다.y

In [2]:
from streamlit import title
from urllib3.contrib.emscripten import response
# requests, beautifulsoup 모듈 설치
!pip install requests beautifulsoup4

ModuleNotFoundError: No module named 'js'

## 웹 페이지 요청 및 파싱

In [5]:
import requests
from bs4 import BeautifulSoup

url = 'https://www.naver.com'

# 일부 사이트는 브라우저가 아닌 요청을 제한할 수 있으므로 User-Agent를 함께 전달한다.
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/125.0 Safari/537.36"
}


try:
    response = requests.get(url, headers=headers, timeout=10)

    # 400,500번대 응답 상태 코드 확인 시 에러
    response.raise_for_status()
except requests.exceptions.Timeout:
    print("시간초과")
except requests.exceptions.RequestException as e:
    print("요청 실패: ",e)
else:
    # 정상 응답 시
    html = response.text
    # print(html)

    # BeautifulSoup 객체를 이용해서 파싱
    soup = BeautifulSoup(html, 'html.parser')
    print(soup.title)









<title>NAVER</title>


## find | find_all

In [6]:
from bs4 import BeautifulSoup
with open('sample.html', 'r', encoding='utf-8') as f:
    html = f.read() # sample.html 내용 읽어오기 -> html모양의 문자열

    #html 문자열 파싱
    soup = BeautifulSoup(html, 'html.parser')
    print(soup.title)

    # find_all : 모든 일치하는 태그 조회
    li_tags = soup.find_all('li')
    # print(li_tags)
    for li in li_tags:
        print(type(li), li.text)

    print("-" *50)

    # find: 일치하는 첫 번째 태그만 조회
    first_li = soup.find('li')
    print("first_li : ", first_li.text)

    # find ('태그명', {속성명:속성값}) : 첫 번째 요소 찾기
    h1_tag = soup.find('h1', {'id': "page-title"})
    print("h1_tag : ", h1_tag)

    # find_all ('태그명', {속성명:속성값}) : 모든 번째 요소 찾기
    section_content_tags = soup.find_all('section', {'class':
    'section-content'})

    for section in section_content_tags:
        print("section.text : ", section.text ) # 내용
        print("section.attrs : ", section.attrs) # 속성

<title>샘플 HTML Page</title>
<class 'bs4.element.Tag'> Section 1
<class 'bs4.element.Tag'> Section 2
<class 'bs4.element.Tag'> Section 3
<class 'bs4.element.Tag'> List item 1
<class 'bs4.element.Tag'> List item 2
<class 'bs4.element.Tag'> List item 3
<class 'bs4.element.Tag'> First item
<class 'bs4.element.Tag'> Second item
<class 'bs4.element.Tag'> Third item
--------------------------------------------------
first_li :  Section 1
h1_tag :  <h1 id="page-title">Welcome to the Sample HTML Page</h1>
section.text :  
Section 1: Introduction

            This is a sample paragraph.
            Bold text
            and
            italic text
            are commonly used to emphasize content. You can also include
            links
            to other pages.
        
Here is an image:


section.attrs :  {'id': 'section1', 'class': ['section-content']}
section.text :  
Section 2: List Examples
Unordered List

List item 1
List item 2
List item 3

Ordered List

First item
Second item
Third it

## select | select_one
- css 선택자(태그명, #, ., > .... 기호)를 이용하여 태그 조회

In [7]:
li_tags = soup.select('li')
# print(li_tags)

first_li = soup.select_one('li')
# print(first_li)

# id(#)를 이용해서 찾기
h1_tag = soup.select_one('#page-title')
# print(h1_tag)

# class(.)를 이용해서 찾기
section_content_tags = soup.select('.section-content')
# print(len(section_content_tags)) #3

# 자식 요소 찾기 (부모 > 자식)
h2_tags = soup.select('.section-content > h2')
# print(h2_tags)

# 후손(하위 모든 요소) 찾기 (부모 후손) (띄어쓰기가 선택자!)
strong_tags = soup.select('.section-content strong')
# print(strong_tags)


# 이전 형제 찾기: find_previous_sibling()
# 다음 형제 찾기: find_next_sibling()
em_tag = soup.select_one('.section-content em')
print(em_tag)
print(em_tag.find_previous_sibling())
print(em_tag.find_next_sibling())

<em>italic text</em>
<strong class="highlight">Bold text</strong>
<a href="https://www.example.com" target="_blank">links</a>


## 네이버 뉴스 검색 결과에서 제목만 스크래핑하기

In [17]:
import requests
from bs4 import BeautifulSoup as bs

query = '인공지능' # 검색어
url = f'https://search.naver.com/search.naver?sm=tab_hty.top&where=news&ssc=tab.news.all&query={query}'

try:
    response = requests.get(url,timeout=10)
    response.raise_for_status()
except requests.exceptions.Timeout:
    print("시간 초과")
except requests.exceptions.RequestException as e:
    print("오류 발생: ", e)
else:
    soup = bs(response.text, 'html.parser')

    # 파싱된 html 코드에서 '뉴스 제목 찾기'
    title_tags = soup.select('.sds-comps-text.sds-comps-text-ellipsis.sds-comps-text-ellipsis-1.sds-comps-text-type-headline1')

    # strip=True -> 좌우 공백 제거
    titles = [tag.get_text(strip=True) for tag in title_tags]

    for title in titles:
        print(title)

KAIST,인공지능최대 난제 '발열' 잡을 냉각기술 확보
SK바이오팜, 바이오 USA서인공지능(AI) 신약개발 전략 공개
“업무 방식 혁신 나서야” 롯데 신동빈 회장,인공지능전환 속도
“사번 001, AI 과장입니다”… SKT ‘인공지능동료’ 파격 실험
경과원, '인공지능(AI) 전문인력 양성 사업' 교육생 60명 모집
인공지능경쟁 ‘전력 인프라·데이터·제도’로 확대
SK AX,인공지능이 기업성장 이끄는 '에이전틱 엔터프라이즈' 시대 연다
인공지능투자 열풍에 반도체 주도...5월 수출물가 11개월째 상승
뷰노, 대한결핵·호흡기학회와 호흡기 의료인공지능(AI) 연구 협력
인공지능·디지털 교육 거점 3곳으로 확대


In [18]:
#  뉴스 검색 결과에서 기사 링크만 스크래핑
a_tags = soup.select(".sds-comps-vertical-layout.sds-comps-full-layout > a:first-of-type")

links = [tag.get('href') for tag in a_tags]

print("수집된 링크 개수: ", len(links))
for link in links:
    print(link)

수집된 링크 개수:  10
https://www.newsis.com/view/NISX20260616_0003670583
https://www.hankyung.com/article/202606168441i
https://www.joongang.co.kr/article/25437169
https://www.munhwa.com/article/11595889?ref=naver
http://www.suwonilbo.kr/news/articleView.html?idxno=316358
https://www.naeil.com/news/read/592078?ref=naver
https://www.aitimes.kr/news/articleView.html?idxno=40529
https://www.m-economynews.com/news/article.html?no=67934
https://www.hankyung.com/article/202606168388i
https://news.kbs.co.kr/news/pc/view/view.do?ncd=8587188&ref=A


In [19]:
# 기사 설명
description_tags = soup.select('span.sds-comps-text.sds-comps-text-ellipsis.sds-comps-text-ellipsis-3.sds-comps-text-type-body1')
descriptions = [span.get_text(' ', strip=True) for span in description_tags]

print(f'수집된 설명 개수: {len(descriptions)}')
print(descriptions)

수집된 설명 개수: 10
['인공지능 (A)I 데이터센터의 냉각전력을 90%나 줄 일 수 있는 발열관리기술이 나왔다. 한국과학기술원(카이스트·KAIST)는 기계공학과 김성진 교수팀과 AX학과 이익진 교수팀이 공동연구를 통해 반도체 칩 내부에 머리카락보다 가는 물길을 새겨 넣는 초고효율 액체 냉각기술을 개발해 AI 반도체의 최대...', 'SK바이오팜은 이번 행사에서 다수의 글로벌 제약사 및 투자자들과의 파트너링 미팅을 통해 신규 협력 기회를 모색하고, 부스를 통해 연구개발과 회사 운영 전반에 걸친 인공지능 (AI) 활용 방향을 소개할 예정이다. BIO USA는 美 바이오협회(BIO)가 주관하는 세계 최대 규모의 바이오 산업 행사다. 올해...', '롯데가 신동빈 롯데그룹 회장을 필두로 전사적인 인공지능 전환(AX, AI Transformation)에 속도를 낸다고 16일 밝혔다. 롯데에 따르면 신 회장은 이달 5~6일 ‘CEO AI 아카데미’에 참여해 바이브 코딩(Vibe coding, 자연어로 요구 사항을 입력하면 AI가 코드를 구현하는 방식)을 기반으로 직접 AI 서비스와 AI...', 'SK텔레콤이 인공지능 (AI) 에이전트에 사번을 부여하고 소속과 직무·권한까지 할당하는 파격적인 실험에 나서 눈길을 끌고 있다. SK텔레콤은 구성원이 AI 전환(AX)에 몰입할 수 있는 환경을 구축하기 위해 이 같은 내용을 담은 ‘AX 혁신 2.0’을 시행한다고 16일 밝혔다. 현장의 업무 효율성을 개선하는 데...', "경기도와 경기도경제과학진흥원(원장 김현곤, 이하 경과원)이 기업 수요에 맞는 디지털 인재를 양성하기 위해 ' 인공지능 (AI) 전문인력 양성 사업' 교육생을 30일까지 모집한다. 이번 사업은 도내 청년들에게 인공지능 (AI) 분야의 실무 역량을 갖춘 전문인력으로 성장할 기회를 제공하고, 도내 산업 전반에...", '글로벌 인공지능 (AI) 경쟁이 모델 성능 경쟁을 넘어 AI를 구동할 전력 인프라와 활용 가능한 데이터, 기업의 투자를 뒷받침할 제도 기반으로

## 스크래핑한 기사를 저장할 클래스

In [21]:
from datetime import datetime

class NaverNews:
    def __init__(self, id: int, title: str, originallink: str, link: str, description: str, pub_date: str,
                 created_at: datetime = None):
        # id는 DB의 auto_increment 기본키를 담기 위한 값이다.
        # 스크래핑 직후에는 아직 DB에 저장되지 않았으므로 None을 사용할 수 있다.
        self.__id = id
        self.__title = title
        self.__originallink = originallink
        self.__link = link
        self.__description = description

        # 네이버 API 응답 key는 pubDate이지만, Python 객체 내부에서는 pub_date 이름을 사용한다.
        self.__pub_date = pub_date
        self.__created_at = created_at

    @classmethod
    def from_api_item(cls, item: dict):
        """네이버 API 응답 item의 pubDate key를 pub_date 속성으로 변환한다."""

        return cls(
            id=None,
            title=item.get('title', ''),
            originallink=item.get('originallink', ''),
            link=item.get('link', ''),
            description=item.get('description', ''),
            pub_date=item.get('pubDate', ''),
        )

    @property
    def id(self):
        return self.__id

    @property
    def title(self):
        return self.__title

    # __title 속성에 대한 setter 메소드
    # 아래 주석을 해제하면 naver_news.title = '새 제목'처럼 값을 변경할 수 있다.
    # @title.setter
    # def title(self, value):
    #     self.__title = value

    @property
    def originallink(self):
        return self.__originallink

    @property
    def link(self):
        return self.__link

    @property
    def description(self):
        return self.__description

    @property
    def pub_date(self):
        """Python 코드에서 사용할 발행일 속성."""

        return self.__pub_date

    @property
    def created_at(self):
        return self.__created_at

    def __repr__(self):
        return f'NaverNews({self.__id}, {self.__title}, {self.__originallink}, {self.__link}, {self.__description}, {self.__pub_date}, {self.__created_at})'

# 스크래핑한 titles, links, descriptions를 묶어서 네이버 뉴스 객체 리스트로 변환

In [23]:
print(len(titles), len(links), len(descriptions))

# zip은 개수가 적은 list를 기준으로 묶는다
news_lines = zip(titles, links, descriptions)

naver_news_list: list[NaverNews] = []

for title, link, description in news_lines:
    naver_news_list.append(
        NaverNews(
            id = None,
            title = title,
            originallink = link,
            link = link,
            description = description,
            pub_date = None
        )
    )


for news in naver_news_list:
    print(news)

10 10 10
NaverNews(None, KAIST,인공지능최대 난제 '발열' 잡을 냉각기술 확보, https://www.newsis.com/view/NISX20260616_0003670583, https://www.newsis.com/view/NISX20260616_0003670583, 인공지능 (A)I 데이터센터의 냉각전력을 90%나 줄 일 수 있는 발열관리기술이 나왔다. 한국과학기술원(카이스트·KAIST)는 기계공학과 김성진 교수팀과 AX학과 이익진 교수팀이 공동연구를 통해 반도체 칩 내부에 머리카락보다 가는 물길을 새겨 넣는 초고효율 액체 냉각기술을 개발해 AI 반도체의 최대..., None, None)
NaverNews(None, SK바이오팜, 바이오 USA서인공지능(AI) 신약개발 전략 공개, https://www.hankyung.com/article/202606168441i, https://www.hankyung.com/article/202606168441i, SK바이오팜은 이번 행사에서 다수의 글로벌 제약사 및 투자자들과의 파트너링 미팅을 통해 신규 협력 기회를 모색하고, 부스를 통해 연구개발과 회사 운영 전반에 걸친 인공지능 (AI) 활용 방향을 소개할 예정이다. BIO USA는 美 바이오협회(BIO)가 주관하는 세계 최대 규모의 바이오 산업 행사다. 올해..., None, None)
NaverNews(None, “업무 방식 혁신 나서야” 롯데 신동빈 회장,인공지능전환 속도, https://www.joongang.co.kr/article/25437169, https://www.joongang.co.kr/article/25437169, 롯데가 신동빈 롯데그룹 회장을 필두로 전사적인 인공지능 전환(AX, AI Transformation)에 속도를 낸다고 16일 밝혔다. 롯데에 따르면 신 회장은 이달 5~6일 ‘CEO AI 아카데미’에 참여해 바이브 코딩(Vibe coding, 자연어로 요구 사항을 입력하면 AI가 코드를 구현하는 방식)을 기반으

## 이미지 스크래핑

In [24]:
import os

parent_dir = './naver_news_images'
os.makedirs(parent_dir, exist_ok=True) # 폴더 없으면 생성

image_tags = soup.select(".sds-comps-base-layout.sds-comps-full-layout img")

for index , image_tag in enumerate (image_tags):
    src = image_tag.get('src') # 이미지 주소 얻어오기

    try:
        image_response = requests.get(src, timeout=10)
        image_response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print("오류 발생 : ", e)

    # 저장할 경로
    file_path = os.path.join(parent_dir,f'{index}.jpg')
    with open(file_path, 'wb') as f:
        f.write(image_response.content)

print("이미지 저장 완료")






이미지 저장 완료
